In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

from src.data import load_fd001, add_rul, drop_cols

In [2]:
import numpy as np
train, test, rul = load_fd001()
train = add_rul(train)
print(train.shape)
print(train.groupby("unit")["RUL"].min().unique())
print(train["RUL"].max())

(20631, 27)
[0]
125


GroupKFold:

In [4]:
from sklearn.model_selection import GroupKFold
features = ["s2", "s3", "s4", "s8", "s11", "s13", "s15", "s17", "s7", "s12", "s20", "s21", "s9", "s14"]
X = train[features]
y = train["RUL"]
groups = train["unit"]
gfk = GroupKFold(n_splits = 10)

In [6]:
splits = list(gfk.split(X,y, groups = groups))
print(len(splits))
print(splits[0])

10
(array([    0,     1,     2, ..., 20628, 20629, 20630], shape=(18559,)), array([ 4565,  4566,  4567, ..., 19008, 19009, 19010], shape=(2072,)))


In [7]:
for fold, (train_idx, val_idx) in enumerate(splits):
    train_units = set(groups.iloc[train_idx])
    val_units = set(groups.iloc[val_idx])
    overlap = train_units & val_units
    print(f"Fold {fold}: {len(val_units)} Triebwerke, {len(val_idx)} Zeilen, Überschneidung {len(overlap)}")

Fold 0: 10 Triebwerke, 2072 Zeilen, Überschneidung 0
Fold 1: 10 Triebwerke, 2057 Zeilen, Überschneidung 0
Fold 2: 10 Triebwerke, 2069 Zeilen, Überschneidung 0
Fold 3: 10 Triebwerke, 2066 Zeilen, Überschneidung 0
Fold 4: 10 Triebwerke, 2066 Zeilen, Überschneidung 0
Fold 5: 10 Triebwerke, 2055 Zeilen, Überschneidung 0
Fold 6: 10 Triebwerke, 2048 Zeilen, Überschneidung 0
Fold 7: 10 Triebwerke, 2056 Zeilen, Überschneidung 0
Fold 8: 10 Triebwerke, 2071 Zeilen, Überschneidung 0
Fold 9: 10 Triebwerke, 2071 Zeilen, Überschneidung 0


Formel für NASA-Score: 
$$s = \begin{cases} e^{-d/13} - 1 & \text{wenn } d < 0 \quad \text{(zu früh)} \\ e^{d/10} - 1 & \text{wenn } d \geq 0 \quad \text{(zu spät)} \end{cases}$$
mit d = Vorhersage - wahre RUL

In [11]:
print(f"10 Zyklen zu früh: {np.exp(10/13)-1:.2f} 30 Zyklen zu früh: {np.exp(30/13)-1:.2f}")
print(f"10 Zyklen zu spät: {np.exp(10/10)-1:.2f} 30 Zyklen zu spät: {np.exp(30/10)-1:.2f}")

10 Zyklen zu früh: 1.16 30 Zyklen zu früh: 9.05
10 Zyklen zu spät: 1.72 30 Zyklen zu spät: 19.09
